In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import pickle
import warnings
import os
warnings.filterwarnings('ignore')

In [67]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from nltk.corpus import stopwords

In [ ]:
ntlk.download('stopwords', queit=True)
from ntlk.corpus import stopwords

In [49]:
os.makedirs('FactCheck_models', exist_ok= True)
os.makedirs('FactCheck_outputs', exist_ok=True)

In [5]:
df = pd.read_csv('dataset/train.csv')

In [9]:
df.tail()

,id,title,author,text,label
20795,20795,Rapper T.I.: Trump a ’Poster Child For White S...,Jerome Hudson,Rapper T. I. unloaded on black celebrities who...,0
20796,20796,"N.F.L. Playoffs: Schedule, Matchups and Odds -...",Benjamin Hoffman,When the Green Bay Packers lost to the Washing...,0
20797,20797,Macy’s Is Said to Receive Takeover Approach by...,Michael J. de la Merced and Rachel Abrams,The Macy’s of today grew from the union of sev...,0
20798,20798,"NATO, Russia To Hold Parallel Exercises In Bal...",Alex Ansary,"NATO, Russia To Hold Parallel Exercises In Bal...",1
20799,20799,What Keeps the F-35 Alive,David Swanson,"David Swanson is an author, activist, journa...",1


In [10]:
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [11]:
df.columns.tolist()

['id', 'title', 'author', 'text', 'label']

In [21]:
df.sum()

id                                                216309600
title     House Dem Aide: We Didn’t Even See Comey’s Let...
author    Darrell LucusDaniel J. FlynnConsortiumnews.com...
text      House Dem Aide: We Didn’t Even See Comey’s Let...
label                                                 10413
dtype: object

In [23]:
df.dtypes

id        int64
title       str
author      str
text        str
label     int64
dtype: object

In [24]:
df.describe()

,id,label
count,20800.000000,20800.000000
mean,10399.500000,0.500625
std,6004.587135,0.500012
min,0.000000,0.000000
25%,5199.750000,0.000000
50%,10399.500000,1.000000
75%,15599.250000,1.000000
max,20799.000000,1.000000


In [27]:
print(f"loaded {len(df)} rows from train.csv")

loaded 20800 rows from train.csv


In [29]:
print(f"columns are {df.columns.tolist()}")

columns are ['id', 'title', 'author', 'text', 'label']


In [33]:
df.isnull().sum()

id           0
title      558
author    1957
text        39
label        0
dtype: int64

In [34]:
df = df.dropna()

In [35]:
df.isnull().sum()

id        0
title     0
author    0
text      0
label     0
dtype: int64

In [37]:
df.duplicated().sum()

np.int64(0)

In [41]:
duplicates = df['text'].duplicated().sum()
print(f"Duplicate texts: {duplicates:,} ({duplicates/len(df)*100:.1f}%)")

   Duplicate texts: 268 (1.5%)


In [42]:
df = df.drop_duplicates(subset=['text'])

In [46]:
print(f"after deduplication: {len(df):,} unique samples")

after deduplication: 18,017 unique samples


##### EDA

In [50]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
label_counts = df['label'].value_counts()

axes[0].pie(label_counts.values, labels=['real news', 'fake news'], 
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
axes[0].set_title('news distribution', fontweight='bold')

sns.countplot(x='label', data=df, ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_xticklabels(['real news', 'fake news'])
axes[1].set_title('sample count', fontweight='bold')
axes[1].set_ylabel('count')

plt.tight_layout()
plt.savefig('FactCheck_outputs/distribution.png', dpi=150)
plt.close()
print("distribution plot saved")

distribution plot saved


In [54]:
df['text_length'] = df['text'].apply(len)
print(f"real news avg length: {df[df['label']==0]['text_length'].mean():.0f} chars")
print(f"fake news avg length: {df[df['label']==1]['text_length'].mean():.0f} chars")

real news avg length: 5199 chars
fake news avg length: 4329 chars


In [63]:
print("real news sample")
real_samples = df[df['label']==0]['text'].head(5)
for i, text in enumerate(real_samples):
    print(f"{i+1}. {text[:100]}")

real news sample
1. Ever get the feeling your life circles the roundabout rather than heads in a straight line toward th
2. In these trying times, Jackie Mason is the Voice of Reason. [In this week’s exclusive clip for Breit
3. PARIS  —   France chose an idealistic, traditional   candidate in Sunday’s primary to represent the 
4. A week before Michael T. Flynn resigned as national security adviser, a sealed proposal was   to his
5. Organizing for Action, the activist group that morphed from Barack Obama’s first presidential campai


In [62]:
print("fake news sample")
fake_samples = df[df['label']==1]['text'].head(5)
for i, text in enumerate(fake_samples):
    print(f"{i+1}. {text[:100]}")

fake news sample
1. House Dem Aide: We Didn’t Even See Comey’s Letter Until Jason Chaffetz Tweeted It By Darrell Lucus o
2. Why the Truth Might Get You Fired October 29, 2016 
The tension between intelligence analysts and po
3. Videos 15 Civilians Killed In Single US Airstrike Have Been Identified The rate at which civilians a
4. Print 
An Iranian woman has been sentenced to six years in prison after Iran’s Revolutionary Guard s
5. The mystery surrounding The Third Reich and Nazi Germany is still a subject of debate between many o


## text preprocessing

In [70]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text))
    text = text.lower()
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 2]
    return ' '.join(words)

df['clean_text'] = df['text'].apply(preprocess_text)

In [74]:
X_train, X_test, Y_train, Y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state = 42, stratify=df['label']
)

In [84]:
## TF-IDF Vectorization

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.85,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"training: {len(X_train):,}")
print(f"test: {len(X_test):,}")

training: 14,413
test: 3,604
